# Assignment 2: Scalable Data Processing

## Yelp Reviews and Authenticity
**_Note_**: See the Assignment 2 PDF for full task descriptions and details!

This project focuses on large-scale data analysis using Apache Spark. You will work with the Yelp Academic Dataset to perform distributed queries, investigate the use of authenticity language in restaurant reviews, and build a machine learning model for rating prediction.

**Tools:** Apache Spark: PySpark, Spark SQL, MLlib, Databricks serverless  

**Submission:**  
- (1) **PDF report**: Export your .ipynb notebook as a PDF.
- (2) **Jupyter Notebook**: Submit your original .ipynb file.

## Dataset
The data you will use is from the [YELP ACADEMIC DATASET](https://business.yelp.com/data/resources/open-dataset/), a dataset of reviews given by yelp users to businesses.

To find the shared folder with the data, you go to left panel and follow the steps:

- Catalog > My organization > workspace > default > yelp_data

In [0]:
# Load the Yelp datasets
# business = spark.read.format("json").load("/Volumes/workspace/default/yelp_data/yelp_academic_dataset_business.json")
# reviews = spark.read.format("json").load("/Volumes/workspace/default/yelp_data/yelp_academic_dataset_review.json")
# users = spark.read.format("json").load("/Volumes/workspace/default/yelp_data/yelp_academic_dataset_user.json")


In [0]:
# # There are 150.346 samples in the business dataframe!
# print(business.count())

# # See the first 5 rows
# business.show(5)

# # Display schema
# business.printSchema()

## Example PySpark Usage: Filtering and Selecting Data

Before starting the assignment, here is a simple example demonstrating how to use Spark DataFrames.

We will:
- Filter businesses whose **name starts with the letter "S"**
- Select the columns: name, city, and stars
- Count how many businesses start with S

Note:
- This example uses the `business` DataFrame.
- We use `F` as a shorthand for `pyspark.sql.functions`, which provides many useful functions for working with DataFrames.


In [0]:
# # Import functions module with alias "F"
# from pyspark.sql import functions as F

# # Filter businesses where the name starts with 'S'
# example_df = business.filter(
#     F.col("name").startswith("S")
# )

# # Select relevant columns and show resulting table
# example_df = example_df.select("name", "city", "stars")
# example_df.show(10, truncate=False)

# # Count how many businesses match the condition
# count_s = example_df.count()
# print("Number of businesses starting with 'S':", count_s)

# Assignment 2

You are now ready to begin the assignment. Please structure your notebook according to the assignment PDF document. 

- **Part A**: DataFrame Queries
- **Part B**: Authenticity Language Analysis
- **Part C**: Rating Prediction with Machine Learning

**Note**: Make sure to follow the specific requirements in the PDF (e.g., no SQL queries in Part A).

For each question:
1. In a **markdown cell**: Add a short description of the task (e.g., *“Q1: Compute the total number of reviews…”*)
2. In a **code cell**: Write your code
3. In a **markdown cell**: Answer the question and interpret the results

In [2]:
import os, sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Stop evt. eksisterende session
try:
    spark.stop()
except NameError:
    pass

spark_temp_dir = Path("spark-temp").resolve()
spark_temp_dir.mkdir(exist_ok=True)

spark = (
    SparkSession.builder
    .appName("YelpLocalAnalysis")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.local.dir", str(spark_temp_dir))
    .getOrCreate()
)

# Bekræft at indstillingen blev brugt
print(spark.sparkContext.getConf().get("spark.local.dir"))

C:\Users\mchrn\ITU\ComputerScienceFolders\3_semester\BDM\BDM-Assignment2\spark-temp


In [3]:
data_dir = Path("data").resolve()
business = spark.read.json(str(data_dir / "yelp_academic_dataset_business.json"))
reviews = spark.read.json(str(data_dir / "yelp_academic_dataset_review.json"))
users = spark.read.json(str(data_dir / "yelp_academic_dataset_user.json"))

print(business.count())
print(reviews.count())
print(users.count())

150346
6990280
1987897


In [ ]:
# Display 5 rows of each data set
business.show(5)
reviews.show(5)
users.show(5)

# Display their schemas to get an overview of the data
business.printSchema()
reviews.printSchema()
users.printSchema()

+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+-------+----------+------------+--------------------+-----------+------------+-----+-----+
|             address|          attributes|         business_id|          categories|         city|               hours|is_open|  latitude|   longitude|                name|postal_code|review_count|stars|state|
+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+-------+----------+------------+--------------------+-----------+------------+-----+-----+
|1616 Chapala St, ...|{NULL, NULL, NULL...|Pns2l4eNsfO8kk83d...|Doctors, Traditio...|Santa Barbara|                NULL|      0|34.4266787|-119.7111968|Abby Rappoport, L...|      93101|           7|  5.0|   CA|
|87 Grasso Plaza S...|{NULL, NULL, NULL...|mpf3x-BjTdTEA3yCZ...|Shipping Centers,...|       Affton|{8:0-18:30, 0:0-0...|      1| 38.551126|  -90.335695|    

## Part A: DataFrame Queries

### **Q1. Compute the total number of reviews for all businesses.**

In [5]:
# Since the business dataset contains a column with review counts, I make a dataframe containing this column and sum it up afterwards.

# Create a new column with the total number of reviews
total_reviews = business.select(F.sum('review_count'))
# Extract the total number of reviews from our new column
print("Answer to Q1 = ", total_reviews.collect()[0]['sum(review_count)'])

Answer to Q1 =  6745508


The total number of reviews for all businesses is 6745508

### **Q2. Find all businesses that have received 5 stars and that have been reviewed by 750 or more users. The output should be a dataframe with columns: name, stars, review count.**

In [ ]:
# I interpret Q2 in the following way: I need to find all business' that have a total star rating of 5
# which I start off by doing:
five_star_business = business.select("business_id", "name", "stars", "review_count").where(F.col("stars") == 5.0)
# I show a few rows to ensure the dataframe looks as I want it
five_star_business.show(5)
print("Amount of businesses that have a star rating of 5.0", five_star_business.count())


# reviewers_df.count()
# reviewers_df.show(5)

+--------------------+--------------------+-----+------------+
|         business_id|                name|stars|review_count|
+--------------------+--------------------+-----+------------+
|Pns2l4eNsfO8kk83d...|Abby Rappoport, L...|  5.0|           7|
|jaxMSoInw8Poo3XeM...|        Adams Dental|  5.0|          10|
|PSo_C1Sfa13JHjzVN...|Indian Walk Veter...|  5.0|          15|
|fvWn8oXXwbj2l79co...|Altitude Trampoli...|  5.0|          30|
|fSCNwMtNNQY9QT69C...|   Sierra Pro Events|  5.0|           7|
+--------------------+--------------------+-----+------------+
only showing top 5 rows
Amount of businesses that have a star rating of 5.0 16307
+--------------------+-----------------------+
|         business_id|calculated_review_count|
+--------------------+-----------------------+
|WKMJwqnfZKsAae75R...|                     40|
|sk2lZI4zmuGAccd3D...|                     33|
|zJErbOQMKX-MwHs_u...|                    222|
|NQhyMw8SOU1HB-V9X...|                     50|
|neL2xrin-uMJl5ABK...|

In [ ]:
# While I could use the review_count column in the business dataframe to look for only businesses with 750 or more reviews:
Q2_v1_answer = business.select("name", "review_count", "stars").where(F.col("stars") == 5.0).where(F.col("review_count") > 750)
print("Q2_v1_answer below")
Q2_v1_answer.show(truncate=False)
# this does not ensure that the 750 reviews are by unique users, and therefore I do something different.

# To count unique users per business_id in the reviews dataset, i can group them by business_id
# and then aggregate it by using the pyspark function count_distinct to count distinct user_id's
# which I collect in a column called unique_user_count
unique_users_per_business = reviews.groupBy("business_id").agg(F.count_distinct("user_id").alias("unique_user_count"))
# unique_users_per_business.show(5)

# I can now join my unique_users_per_business dataframe together with the business dataframe
# where I first filter ensuring that there is atleast 750 unique users that have reviewed the business
# I then join on the business id's and then filter on the stars, so only businesses with 5 stars are left
Q2_v2_answer = (unique_users_per_business
                .where(F.col("unique_user_count") >= 750)
                .join(business.select("business_id", "name", "stars"), on="business_id")
                .where(F.col("stars") == 5.0)
                .select("name", F.col("unique_user_count").alias("review_count"), "stars")
              )
print("Q2_v2_answer below")
Q2_v2_answer.show(truncate=False)

# As seen below it would seem that the review_count in the original business dataset, at least for these 3 businesses actually does
# only have unique user reviews in that metric, as we end up with the same results. 

Q2_v1_answer below
+------------------+------------+-----+
|name              |review_count|stars|
+------------------+------------+-----+
|Blues City Deli   |991         |5.0  |
|Free Tours By Foot|769         |5.0  |
|Carlillos Cocina  |799         |5.0  |
+------------------+------------+-----+

Q2_v2_answer below
+------------------+------------+-----+
|name              |review_count|stars|
+------------------+------------+-----+
|Blues City Deli   |991         |5.0  |
|Carlillos Cocina  |799         |5.0  |
|Free Tours By Foot|769         |5.0  |
+------------------+------------+-----+



For Q2 there are 3 businesses that have received 750 or more reviews that has a 5 star rating and those are:
- Blues City Deli with 991 reviews
- Carlillos Cocina with 799 reviews
- Free Tours By Foot with 769 reviews

### **Q3. Find influencers who have written more than 750 reviews and have an average rating higher than 4 stars. The output should be a dataframe with column: user id.**

In [31]:
# To find my influencers, I select the columns from the user dataset that I need, i.e, user_id, review_count and average_stars
# i then filter on the average_stars and review_count column in accordance with the specification in the question.
influencers = (users
               .select("user_id", "review_count" ,"average_stars")
               .where(F.col("average_stars") > 4.0)
               .where(F.col("review_count") > 750)
              )
# Just to verify that the influencers dataframe looks correct I show 5 rows from it
influencers.show(5)
# Since the answer should be a dataframe only with the column user_id I select that column from my influencer dataset
Q3_answer = influencers.select("user_id")
Q3_answer.show()
print(Q3_answer.count())


+--------------------+------------+-------------+
|             user_id|review_count|average_stars|
+--------------------+------------+-------------+
|MGPQVLsODMm9ZtYQW...|        1807|         4.06|
|RgDVC3ZUBqpEe6Y1k...|        1282|         4.17|
|VHdY6oG2JPVNjihWh...|        2101|         4.22|
|UQFE3BT1rsIYrcDvu...|        1560|         4.17|
|0G-QF457q_0Z_jKqh...|        2009|          4.1|
+--------------------+------------+-------------+
only showing top 5 rows
+--------------------+
|             user_id|
+--------------------+
|MGPQVLsODMm9ZtYQW...|
|RgDVC3ZUBqpEe6Y1k...|
|VHdY6oG2JPVNjihWh...|
|UQFE3BT1rsIYrcDvu...|
|0G-QF457q_0Z_jKqh...|
|ITa3vh5ERI90G_WP4...|
|oW2bCSJ5bIHyRnoZv...|
|13f_vtUZEmlzweL91...|
|UVUIi0q38pAvsfuEE...|
|zDBOdWtl2PsNY38Ie...|
|7Na1pUcEv3oF_QTRw...|
|MUzkXfPS9JaMgJ907...|
|YWFbTcVnun8i03XnE...|
|Ve0LUwcrzxL7w0RYg...|
|i5Cm53q0pmklQsX8s...|
|IzJ36jX6V6ky5BtoN...|
|eTvp_hYnsrI5-ow_s...|
|7PHHxsjyk7I7oM6EN...|
|WWnhqRnWWjutMqh-2...|
|0QdwQLVxZpgy9Qb2Q..

For Q3 the dataframe for influencers can be seen above. There is a total of 1028 users that have made more than 750 reviews with an average star rating above 4

### **Q4. Find the businesses that have been reviewed by more than 5 influencer users. You can use a view created from your answer to Q3. Return a dataframe with columns: name, average rating.**

In [53]:
# I make a new dataframe that contains all reviews by influencers
all_reviews_by_influencers = reviews.join(influencers, on="user_id")

# I now make a dataframe containing the business id and the amount of influencers that have reviewed the business
# by grouping by business_id and aggregrating over distinct user_id (which are only influencers in this dataframe)
reviewed_by_more_than_five = (all_reviews_by_influencers
                              .groupBy("business_id")
                              .agg(F.count_distinct("user_id")
                                   .alias("influencers_reviewed_count")
                                  )
                              ).where(F.col("influencers_reviewed_count") > 5)
# Sanity check to ensure the dataframe contains the information I want
reviewed_by_more_than_five.show(5)

+--------------------+--------------------------+
|         business_id|influencers_reviewed_count|
+--------------------+--------------------------+
|LmRyuir7orBNN43fy...|                         6|
|h-LoRnHxlAx4tS2UJ...|                         7|
|PQYuPr2Dfm2S49EUL...|                         7|
|X9e_btDPux6JGqqw8...|                         9|
|xoVTtQd6xRrFbQcdf...|                         8|
+--------------------+--------------------------+
only showing top 5 rows


In [54]:
# For me there are two interpretations of what the question wants returned. Either it is simply asking for the name and star rating 
# of the businesses that have been reviewed by more than 5 influencers. The code below returns the dataframe for that interpretation
Q4_v1_answer = business.join(reviewed_by_more_than_five, on="business_id").select("name", F.col("stars").alias("average_rating"))
print(Q4_v1_answer.count())
print("Q4_v1_answer below")
Q4_v1_answer.show()

2298
Q4_v1_answer below
+--------------------+--------------+
|                name|average_rating|
+--------------------+--------------+
|Soulshine Pizza F...|           3.5|
|Sanwa Farmers Market|           4.5|
|El Puerto Restaur...|           4.0|
|Koizi Endless Hib...|           3.5|
|Jeremiah's Italia...|           4.5|
|   Josephine Estelle|           4.0|
|Atlantis Purple P...|           3.5|
|                Husk|           4.0|
|               Mixto|           4.0|
|Renaud's Patisser...|           4.0|
|  URBAN Brew and BBQ|           4.0|
|       Tommy DiNic's|           4.0|
|  CALI - Tampa Palms|           4.0|
| Biscuit Love: Gulch|           4.0|
|      Southern Fresh|           4.5|
|ZooTampa at Lowry...|           4.0|
|                Datz|           4.0|
|Mazzaro's Italian...|           4.5|
|  Butcher and Singer|           4.5|
|Skull's Rainbow Room|           4.5|
+--------------------+--------------+
only showing top 20 rows


In [ ]:
# The other interpretation, is that I am being asked to return a dataframe containing the name of any business that have been reviewed
# by more than 5 influencers, and the average of the ratings the influencers have given the business. 
# I do it by taking my dataframe that contains all reviews by influencers. I group it by business id and aggregate the average
# star rating of the influencers reviews if there are more than 5 of them. Finally I join them on the business dataset
# on business_id, and select the name, and the calculated average rating of the influencers reviews for that business.
Q4_v2_answer = (
    all_reviews_by_influencers
    .groupBy("business_id")
    .agg(
        F.count_distinct("user_id").alias("influencers_reviewed_count"),
        F.avg("stars").alias("average_rating")
    )
    .where(F.col("influencers_reviewed_count") > 5)
    .join(business.select("business_id", "name"), on="business_id")
    .select("name", "average_rating")
)
print(Q4_v2_answer.count())
print("Q4_v2_answer below")
Q4_v2_answer.show()

2298
+--------------------+------------------+
|                name|    average_rating|
+--------------------+------------------+
|Soulshine Pizza F...| 3.857142857142857|
|Sanwa Farmers Market|               4.5|
|El Puerto Restaur...|             4.125|
|Koizi Endless Hib...|               3.5|
|Jeremiah's Italia...|               4.7|
|   Josephine Estelle| 4.285714285714286|
|Atlantis Purple P...| 4.111111111111111|
|                Husk| 4.380952380952381|
|               Mixto| 3.909090909090909|
|Renaud's Patisser...| 4.444444444444445|
|  URBAN Brew and BBQ|3.8181818181818183|
|       Tommy DiNic's|             4.375|
|  CALI - Tampa Palms| 4.142857142857143|
| Biscuit Love: Gulch|            4.3125|
|      Southern Fresh|               4.4|
|ZooTampa at Lowry...| 4.230769230769231|
|                Datz| 4.294117647058823|
|Mazzaro's Italian...|4.7727272727272725|
|  Butcher and Singer|4.7272727272727275|
|Skull's Rainbow Room|               4.5|
+--------------------+-------

For Q4 the dataframes can be seen above. There are a total of 2298 businesses that have been reviewed by more than 5 influencers, and by visually inspecting the two dataframes shown, it would seem that the influencers usually giver a higher average rating than the total average rating of the business (which I think makes sense if the context of an influencer is that they are being paid to leave a review). 

### **Q5. Find an ordered list of users based on the average star counts they have given in all their reviews.**

In [68]:
# To find an ordered list of users based on the avg star rating they have given, I select name, user_id and average_stars from
# the users dataset and use orderBy to order them by average_stars first, and then by their name secondly
users_ordered_by_avg_stars = users.select("name", "user_id", "average_stars").orderBy(F.col("average_stars").desc(), "name")

users_ordered_by_avg_stars.show()

+--------+--------------------+-------------+
|    name|             user_id|average_stars|
+--------+--------------------+-------------+
|   Chris|jd83W1Dj3YmkGbhOa...|          5.0|
|     Dan|v52JCl5pIHOuS3cm2...|          5.0|
|'Candice|2A5qlncEkghcVIHWz...|          5.0|
| 'Korbin|aKBOeIg8L4LznQdb6...|          5.0|
| 'Lesley|jyVVjfR5ZeRdrilJv...|          5.0|
|  'Livia|YxXMS5FUu8tOR1ki-...|          5.0|
|   'Mac'|uv666kVtxNpGCnYb3...|          5.0|
|  'Maria|-ihYIrHimiMhcI6my...|          5.0|
|'NaTasha|VHrmC6OVFoHbcGwoK...|          5.0|
|   'Rick|lnGVPT59jQQ7sUJ1x...|          5.0|
|  'Stacy|7YTKfTHt_gzEOSb4L...|          5.0|
|  'Stacy|I4ImVY92guyuTLxFK...|          5.0|
|,Carolyn|zY6p4_BR-dok7ALDd...|          5.0|
|-Kaitlyn|2tC3XVL3e9ODn8Vt4...|          5.0|
|-Timothy|dnqOu6zrUPIlAB2MQ...|          5.0|
|       .|5tc5aSRnh7ZZLPXbz...|          5.0|
|       .|EeBqPMPaLRSV6of6B...|          5.0|
|       .|EGMifkqKCCfCJWGte...|          5.0|
|       .|zJD_SaeapqfyC9sc4...|   

The dataframe users_ordered_by_avg_stars contains all users, and is ordered by average_stars first and then secondly by name.

## Part B: Authenticity Language Analysis


### **E1. What is the percentage of reviews that contain a variant of the word "authentic", and do not contain "inauthentic".**

In [ ]:
# Extract all the text from all reviews and make it lower case as contains() is case sensitive
text = F.lower(F.col("text"))

# Check whether text contains authentic or inauthentic (All variants of authentic contains authentic)
has_authentic = text.contains("authentic")
has_inauthentic = text.contains("inauthentic")

# Only count the instances that contains authentic and not inauthentic
matching = reviews.where(has_authentic & ~has_inauthentic).count()
total = reviews.count()

# Calculate percentage
percentage = (matching / total) * 100

print(f"{matching} of {total} reviews contains a variant of authentic and do not contain inauthentic = {percentage:.2f}%")

124232 of 6990280 reviews contains a variant of authentic and do not contain inauthentic = 1.78%


For E1 out of the 6990280 reviews there was a total of 124232 that contained a variant of authentic, that also did not contain inauthentic. This is 1,78% of the reviews. 

### **E2. How many reviews contain the string "genuine" grouped by business type?**

In [90]:
# Make dataframe that contains a column with business type, where there is an entry for each thing contained in the categories for businesses
# I do this by using the explode function, and splitting on the commas
business_types = (business
                  .where(F.col("categories").isNotNull())
                  .select(
                    "business_id",
                    "name", 
                    F.explode(F.split("categories", r",\s*")).alias("business_type")
                  )
                )
# Sanity check
business_types.show(10, truncate=False)
print(business_types.select("business_type").distinct().count())

+----------------------+------------------------+----------------------------+
|business_id           |name                    |business_type               |
+----------------------+------------------------+----------------------------+
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Doctors                     |
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Traditional Chinese Medicine|
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Naturopathic/Holistic       |
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Acupuncture                 |
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Health & Medical            |
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Nutritionists               |
|mpf3x-BjTdTEA3yCZrAYPw|The UPS Store           |Shipping Centers            |
|mpf3x-BjTdTEA3yCZrAYPw|The UPS Store           |Local Services              |
|mpf3x-BjTdTEA3yCZrAYPw|The UPS Store           |Notaries                    |
|mpf3x-BjTdTEA3yCZrAYPw|The UPS Store           |Mai

In [ ]:
text = F.lower(F.col("text"))

amount_reviews_by_business_type = (reviews
             .where(text.contains("genuine"))
             .select("review_id", "business_id")
             .join(business_types, on="business_id")
             .groupBy("business_type")
             .agg(F.count_distinct("review_id").alias("n_reviews_contains_genuine"))
             .orderBy("n_reviews_contains_genuine", ascending=False)
            )
print("E2_answer below")
amount_reviews_by_business_type.show()
total_reviews_with_genuine = amount_reviews_by_business_type.select(F.sum("n_reviews_contains_genuine").alias("total")).collect()[0]["total"]

# Extract the total number of reviews from our new column
print("Total amount of reviews containing the word genuine = ", total_reviews_with_genuine)

+--------------------+--------------------------+
|       business_type|n_reviews_contains_genuine|
+--------------------+--------------------------+
|         Restaurants|                     18417|
|                Food|                      7529|
|           Nightlife|                      5891|
|                Bars|                      5561|
|      American (New)|                      4032|
|American (Traditi...|                      3739|
|            Shopping|                      3626|
|  Breakfast & Brunch|                      3563|
|Event Planning & ...|                      3105|
|       Beauty & Spas|                      2978|
|    Health & Medical|                      2902|
|          Sandwiches|                      2850|
|             Seafood|                      2204|
|        Coffee & Tea|                      2152|
|          Automotive|                      1939|
|             Italian|                      1917|
|     Hotels & Travel|                      1819|


For E2 when grouping the amount of reviews that contain the word genuine by business type, the result indicates that genuine is much more commonly used when describing a restaurant, than any other type of business. 

### **E3. How many reviews contain the string "legitimate" and how many the string "illegitimate" grouped by type of cuisine?**

In [ ]:
## This was made by using Claude by Anthropic
cuisineGroups = {
    "American": ["American (Traditional)", "American (New)", "Southern", "Soul Food", "Cajun/Creole",
                 "Barbeque", "Smokehouse", "Steakhouses", "Burgers", "Hot Dogs", "Cheesesteaks",
                 "Chicken Wings", "Diners", "Comfort Food"],
    "Canadian": ["Canadian (New)", "Poutineries"],
    "Mexican": ["Mexican", "Tex-Mex", "Tacos", "New Mexican Cuisine"],
    "LatinAmerican": ["Latin American", "Colombian", "Peruvian", "Brazilian", "Argentine", "Venezuelan",
                      "Salvadoran", "Honduran", "Nicaraguan", "Empanadas"],
    "Caribbean": ["Caribbean", "Cuban", "Puerto Rican", "Dominican", "Haitian", "Trinidadian"],
    "Italian": ["Italian", "Pizza", "Pasta Shops", "Sicilian", "Sardinian", "Tuscan", "Calabrian",
                "Roman", "Cucina campana"],
    "French": ["French", "Brasseries", "Creperies"],
    "Iberian": ["Spanish", "Portuguese", "Iberian", "Basque", "Tapas/Small Plates", "Tapas Bars"],
    "German": ["German", "Austrian"],
    "BritishIrish": ["British", "Irish", "Irish Pub", "Scottish", "Fish & Chips"],
    "EasternEuropean": ["Eastern European", "Polish", "Russian", "Ukrainian", "Czech", "Hungarian",
                        "Serbo Croatian", "Georgian", "Uzbek"],
    "OtherEuropean": ["Modern European", "Belgian", "Scandinavian"],
    "Mediterranean": ["Mediterranean", "Greek"],
    "MiddleEastern": ["Middle Eastern", "Lebanese", "Turkish", "Persian/Iranian", "Arabic", "Israeli",
                      "Syrian", "Armenian", "Afghan", "Kebab", "Falafel", "Pita", "Donairs", "Lahmacun"],
    "African": ["African", "Ethiopian", "Moroccan", "Egyptian", "Senegalese", "Somali", "South African"],
    "Chinese": ["Chinese", "Cantonese", "Szechuan", "Dim Sum", "Hot Pot", "Taiwanese", "Shanghainese",
                "Hakka", "Hainan", "Fuzhou", "Dumplings", "Hong Kong Style Cafe"],
    "Japanese": ["Japanese", "Sushi Bars", "Ramen", "Izakaya", "Teppanyaki", "Japanese Curry",
                 "Conveyor Belt Sushi", "Tonkatsu", "Donburi"],
    "Korean": ["Korean"],
    "Thai": ["Thai"],
    "Vietnamese": ["Vietnamese"],
    "SoutheastAsian": ["Filipino", "Malaysian", "Indonesian", "Laotian", "Cambodian", "Burmese", "Singaporean"],
    "IndianSouthAsian": ["Indian", "Pakistani", "Bangladeshi", "Himalayan/Nepalese", "Sri Lankan"],
    "AsianFusion": ["Asian Fusion", "Pan Asian", "Oriental", "Mongolian"],
    "HawaiianPacific": ["Hawaiian", "Poke", "Guamanian", "Australian"],
    "Seafood": ["Seafood"],
    "VegetarianVegan": ["Vegetarian", "Vegan", "Live/Raw Food"]
}

In [ ]:
# # Find the business_ids of all businesses that have "Restaurants" among their categories
restaurant_ids = (
    business_types
    .where(F.col("business_type") == "Restaurants")
    .select("business_id")
)

# Get every category of those restaurants (the cuisine is listed here, next to "Restaurants"),
# and count how many restaurants have each category. Renamed to yelp_category so it can be
# joined with cuisine_map. Note that the result also includes non-cuisine categories such as Food and Bars.
restaurant_categories = (
    business_types
    .join(restaurant_ids, "business_id")
    .groupBy("business_type").count()
    .withColumnRenamed("business_type", "yelp_category")
)

restaurant_categories.orderBy(F.desc("count")).show(150, truncate=False)

+-------------------------+-----+
|yelp_category            |count|
+-------------------------+-----+
|Restaurants              |52268|
|Food                     |15472|
|Nightlife                |8723 |
|Sandwiches               |8366 |
|Bars                     |8337 |
|American (Traditional)   |8139 |
|Pizza                    |7093 |
|Fast Food                |6472 |
|Breakfast & Brunch       |6239 |
|American (New)           |6097 |
|Burgers                  |5636 |
|Mexican                  |4600 |
|Italian                  |4573 |
|Coffee & Tea             |4053 |
|Seafood                  |3539 |
|Chinese                  |3169 |
|Event Planning & Services|3068 |
|Salad                    |3064 |
|Chicken Wings            |2966 |
|Cafes                    |2756 |
|Delis                    |2393 |
|Caterers                 |2059 |
|Specialty Food           |2011 |
|Bakeries                 |1889 |
|Desserts                 |1838 |
|Japanese                 |1830 |
|Sports Bars  

In [ ]:
# Create a dataframe containing a mapping of the category from the dataset, mapped to a cuisine group
rows = [(category, group) for group, categories in cuisineGroups.items() for category in categories]
cuisine_map = spark.createDataFrame(rows, ["yelp_category", "cuisine_group"])
cuisine_map.show(50, truncate=False)

+----------------------+-------------+
|yelp_category         |cuisine_group|
+----------------------+-------------+
|American (Traditional)|American     |
|American (New)        |American     |
|Southern              |American     |
|Soul Food             |American     |
|Cajun/Creole          |American     |
|Barbeque              |American     |
|Smokehouse            |American     |
|Steakhouses           |American     |
|Burgers               |American     |
|Hot Dogs              |American     |
|Cheesesteaks          |American     |
|Chicken Wings         |American     |
|Diners                |American     |
|Comfort Food          |American     |
|Canadian (New)        |Canadian     |
|Poutineries           |Canadian     |
|Mexican               |Mexican      |
|Tex-Mex               |Mexican      |
|Tacos                 |Mexican      |
|New Mexican Cuisine   |Mexican      |
|Latin American        |LatinAmerican|
|Colombian             |LatinAmerican|
|Peruvian              |L

In [ ]:
# Check which categories in the cuisine_map that is not present among the Yelp restaurant categories
# (left_anti keeps only rows from cuisine_map without a match).
cuisine_map.join(restaurant_categories, "yelp_category", "left_anti") \
    .orderBy("cuisine_group").show(100, truncate=False)

+-------------+-------------+
|yelp_category|cuisine_group|
+-------------+-------------+
+-------------+-------------+



In [124]:
# Make a dataframe with a business_id matched to a cuisine group. 
# Distinct to ensure a business is not counted several times for the same cuisine
business_cuisines = (business_types
    .join(cuisine_map, business_types.business_type == cuisine_map.yelp_category)
    .select("business_id", "cuisine_group")
    .distinct()
)
# Sanity check
business_cuisines.show(5)

+--------------------+-------------+
|         business_id|cuisine_group|
+--------------------+-------------+
|2jFOIBICinZkepcO6...|     American|
|UYcXDGDdH4niV1WCn...|     American|
|OdpXHSWneuD2BONO5...|     American|
|VCyFKtBsba_FYwq35...|     American|
|oKvzmVhA1dKf-UARX...|     American|
+--------------------+-------------+
only showing top 5 rows


In [ ]:
text = F.lower(F.col("text"))

has_legitimate = text.contains("legitimate")
has_illegitimate = text.contains("illegitimate")

amount_reviews_contains_legitimate = (reviews
                                      .where(has_legitimate & ~has_illegitimate)
                                      .select("review_id", "business_id")
                                      .join(business_cuisines, on="business_id")
                                      .groupBy("cuisine_group")
                                      .agg(F.count_distinct("review_id").alias("n_reviews_contains_legitimate"))
                                      .orderBy("n_reviews_contains_legitimate", ascending=False)
                                    )
print("E3_answer below")
amount_reviews_contains_legitimate.show()
total_reviews_with_legitimate = amount_reviews_contains_legitimate.select(F.sum("n_reviews_contains_legitimate").alias("total")).collect()[0]["total"]

# Extract the total number of reviews from our new column
print("Total amount of reviews containing the word legitimate = ", total_reviews_with_legitimate)


E2_answer below
+----------------+-----------------------------+
|   cuisine_group|n_reviews_contains_legitimate|
+----------------+-----------------------------+
|        American|                         1352|
|         Italian|                          419|
|         Mexican|                          305|
|         Seafood|                          304|
|        Japanese|                          202|
| VegetarianVegan|                          183|
|         Chinese|                          168|
|     AsianFusion|                          126|
|            Thai|                           86|
|   Mediterranean|                           75|
|          French|                           73|
|   LatinAmerican|                           72|
|      Vietnamese|                           63|
|         Iberian|                           62|
|   MiddleEastern|                           57|
|          Korean|                           49|
|    BritishIrish|                           46|
|   

In [130]:
text = F.lower(F.col("text"))

# \b (word boundary) ensures "illegitimate" is not also matched as "legitimate"
has_legitimate = text.rlike(r"\blegitimate")
has_illegitimate = text.contains("illegitimate")

# Flag each review for the two words, keep only the relevant reviews and join with the cuisine groups.
# business_cuisines is distinct per (business_id, cuisine_group), so a plain sum does not count a review twice in the same group
E3_answer = (reviews
             .select("business_id",
                     has_legitimate.cast("int").alias("legit"),
                     has_illegitimate.cast("int").alias("illegit"))
             .where((F.col("legit") == 1) | (F.col("illegit") == 1))
             .join(business_cuisines, on="business_id")
             .groupBy("cuisine_group")
             .agg(F.sum("legit").alias("n_reviews_contains_legitimate"),
                  F.sum("illegit").alias("n_reviews_contains_illegitimate"))
             .orderBy("n_reviews_contains_legitimate", ascending=False)
            )
print("E3_answer below")
E3_answer.show()

# The sum over cuisine groups is higher than the real number of reviews, since a business can belong to several groups.
# Therefore the real totals are taken directly from the reviews dataframe
totals = reviews.agg(F.sum(has_legitimate.cast("int")).alias("legit_total"),
                     F.sum(has_illegitimate.cast("int")).alias("illegit_total")).collect()[0]

print("Total amount of reviews containing the word legitimate = ", totals["legit_total"])
print("Total amount of reviews containing the word illegitimate = ", totals["illegit_total"])

E3_answer below
+----------------+-----------------------------+-------------------------------+
|   cuisine_group|n_reviews_contains_legitimate|n_reviews_contains_illegitimate|
+----------------+-----------------------------+-------------------------------+
|        American|                         1351|                             13|
|         Italian|                          419|                              6|
|         Mexican|                          305|                              5|
|         Seafood|                          303|                              2|
|        Japanese|                          202|                              0|
| VegetarianVegan|                          183|                              0|
|         Chinese|                          168|                              1|
|     AsianFusion|                          126|                              0|
|            Thai|                           86|                              0|
|   Mediterr

### **E4. Is there a difference in the amount of authenticity language used in the different areas? (e.g., by state, north/south, urban/rural)**

### **E5. How do the percentages of authenticity-related words and negative words differ across South American, Asian, and European restaurants, and by how much? What other categories could be compared?**

## Part C: Rating Prediction with Machine Learning